# 第5章 モデルに点数をつける：損失とパープレキシティ

確率を掛けたら0になった事件と、対数という救世主

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：言語モデルの成績を1つの数で表す「損失（loss）」の計算機と、これから全章で使う「記録表」
- **身につく数学・理論**：訓練データと検証データ（過去問と本番） ／ 確率の掛け算とアンダーフロー ／ 対数 $\log$ と、その3つの性質 ／ 負の対数尤度＝損失（驚きの平均） ／ スムージングとハイパーパラメータ ／ パープレキシティ（何択で迷っているか）
- **新しく使うPython・PyTorch**：`math.log` ／ 関数に初期値つきの引数 ／ `float('inf')`（無限大） ／ `matplotlib` でグラフを描く

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 60〜90分 |
| 難易度 | ★★☆☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 前の章のbigramモデルは「なんとなく漱石っぽい」文章を書きました。でも「なんとなく」では、モデルを改良しても良くなったのか分かりません。この章では、言語モデルの成績を**1つの数**で表す方法を作ります。この数（**損失**）は、この先のすべての章で、モデルの進化を測るものさしになります。

> **🔗 これまでの復習とつながり：第3章・第4章とのつながり**
>
> - **第3章**：どんな数も正の値に変える指数関数 $e^x$ と、合計1の確率分布を作る `softmax` を学びました。
> - **第4章**：『こころ』から隣接ペアを数えて条件付き確率 $P(b \mid a)$ を作る bigram モデルを完成させ、データに出てこない並び（スパース性）という弱点を見つけました。
>
> この章では、条件付き確率 $P(b \mid a)$ を使ってモデルに客観的な点数をつける仕組み（**対数・負の対数尤度・損失・パープレキシティ**）を作ります。第3章の $e^x$ のちょうど逆の道具である **対数 $\log$** が主役として登場します！

**ミナト**：点数のつけ方って、生成した文章を人間が読んで「80点！」とかじゃダメなんですか？

**ボンゴ先生**：それも大事な方法だ（第20章でやる）。でも、学習中のモデルは1秒に何百回も成績を知りたがる。毎回キミが読んで採点していたら、キミが先に倒れる。だから「計算だけで出せる点数」が必要なんだ。

## 準備：『こころ』を読み込む

第4章で作った掃除の関数を、ダウンロードまで含めて1つにまとめておきます。この**準備セル**は、第10章まで毎回最初に実行します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re, math, urllib.request
from collections import Counter

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
print(len(text), "文字")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
162467 文字
```

</details>

## 過去問と本番：データを分ける

モデルの点数をつける前に、大事な準備があります。データを2つに分けるのです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
n = int(len(text) * 0.9)
train_text = text[:n]   # 前の90%：学習用（数える用）
val_text = text[n:]     # 後ろの10%：点数をつける用
print("訓練データ：", len(train_text), "文字")
print("検証データ：", len(val_text), "文字")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
訓練データ： 146220 文字
検証データ： 16247 文字
```

</details>

> **💡 理論のポイント：なぜ分けるの？**
>
> テスト勉強で、過去問の**答えを丸暗記**した人を想像してください。同じ過去問なら満点ですが、本番で少し違う問題が出たら解けません。
>
> モデルも同じです。数えるのに使った文章で点数をつけると、「丸暗記」がうまいほど高得点になってしまいます。本当に知りたいのは「**見たことのない文章**でも、次の文字を当てられるか」です。
>
> そこで、データを分けます。
>
> - **訓練データ**（training data）：モデルが学ぶ（数える）ためのデータ。過去問
> - **検証データ**（validation data）：点数をつけるためだけのデータ。モデルには絶対に見せない。本番の模試
>
> さらに研究では、最後の最後に一度だけ使う **テストデータ** をもう1つ取っておくこともあります（第20章）。

訓練データだけでbigramを数え直します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
pairs = Counter(zip(train_text, train_text[1:]))
firsts = Counter(train_text[:-1])

def prob(a, b):
    """P(b | a)：「a」の次が「b」である確率"""
    if firsts[a] == 0:
        return 0.0
    return pairs[(a, b)] / firsts[a]

print("P(生|先) =", prob("先", "生"))
print("P(は|生) =", prob("生", "は"))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
P(生|先) = 0.8857566765578635
P(は|生) = 0.23097112860892388
```

</details>

#### 📝 確認問題（`ch05-q01` · Lv.2 · 訓練データと検証データを分ける理由（丸暗記の防止））

モデルの成績を測るとき、全テキストで数えて同じテキストで採点するのではなく、あらかじめ**訓練データ（90%）**と**検証データ（10%）**に分ける最大の理由はどれですか？

- **a.** 検証データの方が文章が短く10%しかないため、確率の掛け算をしてもアンダーフローが起きなくなるため
- **b.** 学習に使った文章で採点すると「丸暗記」したモデルほど高得点になり、未知の文章への予測力が測れないため
- **c.** モデルの学習には全体の90%だけで十分なので、残りの10%は前処理ミスとして切り捨てて破棄するため
- **d.** Pythonの `Counter` は15万文字を超えるとメモリ上限に達して正しく集計できなくなってしまうため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「過去問の答えを丸暗記した受験生」に、まったく同じ過去問を出して実力を測れるかどうか考えましょう。
- **ヒント 2**：本当に知りたいのは「見たことのない文章（未知データ）に対する予測力＝汎化性能」です。
- **正解**：`b`

**考え方**：言語モデルの目的は「未知の文章でも次の文字を当てられること（汎化）」です。モデルに一切見せていない検証データで採点することで、丸暗記（過学習）を見抜けます。

1. 訓練データ（90%）＝過去問：カウントや重み学習に使う。
1. 検証データ（10%）＝本番の模試：学習には絶対に使わず、成績比較にだけ使う。
1. 未知データへの実力（汎化性能）を正しく測れる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q01

</details>

#### 📝 確認問題（`ch05-q02` · Lv.2 · train_text[:-1] の役割と未出現文字のゼロ除算ガード）

本文では分母のカウントに `Counter(train_text[:-1])` を使っています。末尾の1文字を除く理由を確認しましょう。次のコードを実行すると何が表示されますか？

```python
from collections import Counter
s = "あいあ"
pairs = Counter(zip(s, s[1:]))
firsts = Counter(s[:-1])
print(firsts["あ"], pairs[("あ", "い")] / firsts["あ"])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`s[:-1]` は末尾の1文字を除いたスライスです。`"あいあ"` の末尾を除くとどの文字列になるか確認しましょう。
- **ヒント 2**：`s[:-1]` は `"あい"` なので `firsts["あ"]` は `1`、ペア `("あ", "い")` も `1` です。Pythonの `/` は小数を返します。
- **期待される出力**：`1 1.0`

**考え方**：末尾の「あ」には次の文字が存在しないため、`s` 全体で数えると分母が2になり $P(\text{い} \mid \text{あ}) = 1/2$ となって合計が1になりません。`s[:-1]` で末尾を除くことで分母と分子の合計が一致します。

1. `s = "あいあ"` → `s[:-1] = "あい"`。
1. `firsts["あ"]` は `1`（末尾の「あ」は数えない）。
1. `pairs[("あ", "い")]` は `1` → `1 / 1 = 1.0`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q02

</details>

## 文章の確率を計算する

モデルが「この文章はありそうだ」と思うほど、良いモデルのはずです。では、文章全体の確率（同時確率）はどう計算するのでしょう？ 数学的に**厳密な等式**と、bigramモデルの**近似**をしっかり区別して押さえましょう。

#### 📐 数学のポイント：厳密な「確率の連鎖律」 vs bigramの「マルコフ近似」

任意の文字列 $(x_1, x_2, x_3)$ が続けて出てくる確率は、条件付き確率の定義から**常に厳密に**次の掛け算に分解できます（**確率の連鎖律**、chain rule of probability）。

$$P(x_1, x_2, x_3) = P(x_1) \cdot P(x_2 \mid x_1) \cdot P(x_3 \mid x_1, x_2)$$

「先生は」なら、厳密には $P(\text{先生は}) = P(\text{先}) \cdot P(\text{生} \mid \text{先}) \cdot P(\text{は} \mid \text{先生})$ です。本物のLLM（GPTなど）は、3文字目を予測するときに**それまでの全文脈 $(x_1, x_2)$ を条件にする**この厳密な連鎖律をそのまま使います。

一方、bigramモデルは「直前の1文字だけを見る」という**マルコフ近似**（$P(x_3 \mid x_1, x_2) \approx P(x_3 \mid x_2)$）を置きます。

$$P_{\text{bigram}}(x_1, x_2, x_3) = P(x_1) \cdot P(x_2 \mid x_1) \cdot P(x_3 \mid x_2)$$

最初の1文字 $P(\text{先})$ を除いた遷移確率の積で「先生は」を計算すると、bigramの近似値は次のようになります。

$$P_{\text{bigram}}(\text{生}, \text{は} \mid \text{先}) = P(\text{生} \mid \text{先}) \times P(\text{は} \mid \text{生}) \approx 0.886 \times 0.231 \approx 0.205$$

**区別が大切**：上の $P(x_1, x_2, x_3) = P(x_1)P(x_2 \mid x_1)P(x_3 \mid x_1, x_2)$ は**常に成り立つ数学の恒等式**、$P(x_3 \mid x_1, x_2) \approx P(x_3 \mid x_2)$ は**計算を簡単にするためのモデル上の近似**です。

では、訓練データの最初の数百文字で bigram の遷移確率を掛け続けてみましょう。どのペアも訓練データで数えたものなので、確率が0になることはないはずです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
p = 1.0
for i, (a, b) in enumerate(zip(train_text, train_text[1:])):
    p = p * prob(a, b)
    if i in [10, 50, 100, 150, 200, 300]:
        print(f"{i:4d}文字目まで：確率 {p}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
10文字目まで：確率 2.999105026951313e-16
  50文字目まで：確率 2.7431965930303487e-71
 100文字目まで：確率 5.506246902145188e-135
 150文字目まで：確率 1.2818683751043334e-213
 200文字目まで：確率 2.0794249766405372e-279
 300文字目まで：確率 0.0
```

</details>

**ボンゴ先生**：事件だ！ どのペアも確率は0じゃないのに、掛け算を続けたら、300文字目でとうとう**ぴったり0**になってしまった。200文字目の `e-279` は「10のマイナス279乗」、つまり小数点の後に0が278個並ぶ数だ。宇宙にある原子の数（だいたい $10^{80}$ 個）で1を割っても、まだまだこんなに小さくはならない。

> **📘 基本情報とのつながり：アンダーフロー**
>
> コンピュータの小数（**浮動小数点数**）には、表せる大きさに限界があります。Pythonの `float`（倍精度、64ビット）では、だいたい $10^{-308}$ より小さい数はまともに表せず、最後は0になってしまいます。これを **アンダーフロー** と言います。基本情報の「浮動小数点数の表現」で、指数部のビット数が限られていることを習いましたね。それがここで牙をむいたのです。

0になってしまうと、どのモデルの確率も「0」で、比べようがありません。どうしましょう？

#### 📝 確認問題（`ch05-q03` · Lv.2 · 文章全体の同時確率の掛け算とアンダーフロー）

訓練データでどのペアの確率も0より大きいのに、300文字ほど掛け算し続けると Python が `0.0` と表示してしまう現象の原因と名称として正しいものはどれですか？

- **a.** **アンダーフロー**：1未満の確率を何百回も掛け、浮動小数点数の下限（約 $10^{-308}$）を下回り0に丸められた
- **b.** **ループ上限リセット**：Pythonの `for` 文は256回を超えると累積変数を自動で `0.0` に初期化する仕様がある
- **c.** **オーバーフロー**：確率の掛け算を何百回も繰り返すうちに値が大きくなりすぎて、浮動小数点数の上限を超えてしまった
- **d.** **スパース性**：訓練データ自身の文章なのに、実際には確率0の未出現ペアが途中に1つ以上紛れ込んでいた

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：1より小さい数を何度も掛け続けると、値は大きくなりますか、小さくなりますか？
- **ヒント 2**：平均 `0.05` を300回掛けると $(0.05)^{300} \approx 10^{-390}$ となり、64ビット小数の下限（約 $10^{-308}$）を下回ります。
- **正解**：`a`

**考え方**：64ビット浮動小数点数が表現できる正の数の下限は約 $10^{-308}$（非正規化数を含めても約 $5 \times 10^{-324}$）です。長い文章の確率積は簡単にこれを下回って `0.0` に潰れてしまいます。

1. 1文字ごとの確率はおおよそ $0.05$ 前後。
1. 200文字で $10^{-279}$ 付近まで縮小。
1. 300文字で $10^{-308}$ を大きく下回り、コンピュータ上は完全に `0.0`（アンダーフロー）になる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q03

</details>

## 救世主：対数（log）

ここで登場するのが **対数**（たいすう、logarithm）です。記号を丸暗記するのではなく、「なぜ対数が必要なのか」から9ステップでたどりましょう。

#### 📐 数学のポイント：対数 $\log$：9ステップで理由から理解する

**① 解決したい具体的な問題**：何千文字もの確率の掛け算 $p_1 \times p_2 \times \dots \times p_N$ を、0に潰れず（アンダーフローせず）、かつ**モデル間の大小関係を変えずに**計算したい。

**② 素朴な掛け算ではなぜダメなのか？**：確率は必ず $0 \le p \le 1$ なので、掛けるたびに桁が急減します。300文字も掛けると $10^{-308}$ を下回り、64ビット浮動小数点数（`float`）では `0.0` に丸められて全モデルの点数が同じ `0.0` になってしまいます。

**③ 新しい発想**：「数そのもの」ではなく、**「ある基準の数を何回掛けたか（指数の肩の数）」**で数える世界へ引っ越す。指数の世界では**掛け算が足し算に化ける**（$10^2 \times 10^3 = 10^{2+3}$）ので、何万回足しても0に潰れません！

**④ 記号の読み方表**：
| 記号 | 読み方 | 意味 |
|---|---|---|
| $\log_{10} x$ | 「ログ10の $x$」 | $10$ を何回掛けたら $x$ になるか（常用対数） |
| $\log x$（または $\ln x$） | 「ログ $x$」（自然対数） | 第3章の $e \approx 2.718$ を何回掛けたら $x$ になるか。Pythonは `math.log(x)` |
| $e^x$ と $\log x$ | 互いに逆の操作 | $\log(e^x) = x$、$e^{\log y} = y$（$y > 0$） |

**⑤ 小さな数字で手計算**：$\log_{10} 1000$ は「10を何回掛けたら1000になるか」という意味で、答えは `3` です（$10 \times 10 \times 10 = 1000$）。

| $x$ | 0.001 | 0.01 | 0.1 | 1 | 10 | 100 | 1000 |
|---|---|---|---|---|---|---|---|
| $\log_{10} x$ | -3 | -2 | -1 | 0 | 1 | 2 | 3 |

1より小さい数はマイナスになります（`0.01` は「10で2回割った」＝ $10^{-2}$ なので `-2`）。掛け算も検算してみましょう：$0.1 \times 0.01 = 0.001$ ですが、対数にすると $(-1) + (-2) = -3 = \log_{10}(0.001)$ とぴったり一致します！

**⑥ 一般式（自然対数）**：機械学習では10の代わりに $e \approx 2.718$ を底にした **自然対数** $\log$ を使います（第3章の `softmax` に出てくる $e^x$ のちょうど逆関数だからです）。
$$\log(p_1 \times p_2 \times \dots \times p_N) = \log p_1 + \log p_2 + \dots + \log p_N = \sum_{i=1}^{N} \log p_i$$

**⑦ Pythonでの検算**（下のセルで実際に確かめます）：`math.log(math.e) == 1.0`、`math.log(1) == 0.0`、`math.log(0.5) ≈ -0.693`。

**⑧ よくある誤解**：$\log(a + b) = \log a + \log b$ ではありません！ 足し算に分解できるのは**掛け算** $\log(a \times b) = \log a + \log b$ のときだけです。また $\log 0$ は定義できず（$-\infty$ に発散）、Pythonではエラーになります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(math.log10(1000), math.log10(0.01))
print(math.log(math.e), math.log(1))
print(math.log(0.5), math.log(0.01))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
3.0 -2.0
1.0 0.0
-0.6931471805599453 -4.605170185988091
```

</details>

> **💡 理論のポイント：対数の3つの性質（これだけ覚えればOK）**
>
> 1. **掛け算が足し算になる**：$\log(a \times b) = \log a + \log b$
> 2. **1は0になる**：$\log 1 = 0$
> 3. **1より小さい数はマイナス**、0に近づくほどどんどんマイナスに大きくなる
>
> 性質1がすごいのです。確率の**掛け算**を、対数の**足し算**に変えられます。足し算なら、何千回くり返しても0にはなりません（-1000とか-5000とかになるだけ）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
log_p = 0.0
for i, (a, b) in enumerate(zip(train_text, train_text[1:])):
    log_p = log_p + math.log(prob(a, b))
    if i in [10, 50, 100, 200, 300]:
        print(f"{i:4d}文字目まで：log確率 {log_p:.1f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
10文字目まで：log確率 -35.7
  50文字目まで：log確率 -162.5
 100文字目まで：log確率 -309.1
 200文字目まで：log確率 -641.7
 300文字目まで：log確率 -951.2
```

</details>

もう0になりません。事件解決です。

#### 📝 確認問題（`ch05-q04` · Lv.2 · 対数の手計算（掛け算→足し算、10のべき乗））

$\log(0.5) \approx -0.693$ を使って、$\log(0.5 \times 0.5 \times 0.5)$ の値を小数第3位まで求めてください（誤差 $\pm 0.02$ まで正解）。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：対数の最も大切な性質「掛け算の対数＝対数の足し算」を使いましょう。
- **ヒント 2**：$\log(0.5 \times 0.5 \times 0.5) = \log 0.5 + \log 0.5 + \log 0.5$ に $\log(0.5) \approx -0.693$ を代入して計算します。
- **正解**：`-2.079`

**考え方**：$\log(a \times b \times c) = \log a + \log b + \log c$ より、$-0.693 \times 3 = -2.079$ になります。

1. 性質 $\log(a \times b) = \log a + \log b$ を適用する。
1. $\log(0.5 \times 0.5 \times 0.5) = \log 0.5 + \log 0.5 + \log 0.5$。
1. $-0.693 \times 3 = -2.079$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q04

</details>

#### 📝 確認問題（`ch05-q05` · Lv.2 · 対数の3つの性質と、なぜ対数でアンダーフローが解決するのか）

長い文章の確率 $p_1 \times p_2 \times \dots \times p_N$ の代わりに $\log p_1 + \log p_2 + \dots + \log p_N$ を計算することで、モデルの優劣を変えずにアンダーフローを防げる理由の組み合わせとして正しいものはどれですか？

- **a.** $\log 0 = 0$ が成り立つため、確率0の未出現ペアが混ざっていてもエラーにならず安全に足し算できるから
- **b.** ①掛け算が足し算に変わり0に潰れず、② $\log$ は単調増加なのでモデル間の順位も保たれる
- **c.** どんな確率に対しても $\log$ をとると必ず $0$ から $1$ の間の正の数に収まり、計算が安定するようになるから
- **d.** 対数には $\log(a + b) = \log a \times \log b$ という性質があり、足し算を掛け算に直せるようになるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：対数に変えても「モデルの優劣が変わらない理由」と「アンダーフローしない理由」の2つに注目しましょう。
- **ヒント 2**：① 掛け算→足し算の変換、② 単調増加性（大きい確率ほど対数も大きい）、の2条件を正しく述べたものを選びます。
- **正解**：`b`

**考え方**：対数は指数関数 $e^x$ の逆関数であり、正の数の掛け算を足し算に変換しつつ大小関係を厳密に保ちます。そのため「同時確率を最大化すること」と「対数確率の和を最大化すること」は数学的に完全に同じ問題になります。

1. 掛け算→足し算：300文字かけても $10^{-390}$ ではなく `-951.2` のように扱いやすい数になる。
1. 単調増加：$P_A > P_B \iff \log P_A > \log P_B$ なので、最良のモデルが変わらない。
1. また $\log(e^x) = x$ なので、第3章の softmax（$e^z$）とも相性抜群。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q05

</details>

## 損失：驚きの平均

「log確率の合計」は文章が長いほどマイナスに大きくなるので、長さの違う文章どうしを比べにくいです。そこで、**1文字あたりの平均**にします。さらに、マイナスの数は扱いにくいので、**符号をひっくり返して**プラスにします。これが言語モデルの点数、**損失**（loss）です。

$$L = -\frac{1}{N}\sum_{i=1}^{N} \log p_i = -\frac{1}{N}\sum_{i=1}^{N} \log P(\text{次の文字}_i \mid \text{今の文字}_i)$$

#### 📐 数学のポイント：なぜ $1-p$ ではなく $-\log p$ なのか？：9ステップで理由から理解する

**① 解決したい具体的な問題**：正解文字に対する予測確率 $p_1, p_2, \dots, p_N$ から、「外したほど大きい罰（損失）」を1つの数 $L$ にまとめたい。

**② 素朴な方法（$1 - p$ や $(1 - p)^2$）ではなぜダメなのか？**
「外れ度合いなら $1 - p$ や $(1 - p)^2$ でよさそう」と思えますよね。実際に極端な外れ値で比べてみましょう。

| モデルが正解につけた確率 $p$ | $1 - p$ | $(1 - p)^2$ | $-\log p$ |
|---|---|---|---|
| $1.0$（完璧に的中） | $0.00$ | $0.00$ | $0.00$ |
| $0.5$（五分五分） | $0.50$ | $0.25$ | $0.69$ |
| $0.01$（1%と軽視） | $0.99$ | $0.9801$ | $4.61$ |
| $0.0001$（0.01%と決めつけて大外れ） | $0.9999$ | $0.9998$ | $9.21$ |
| $0$（絶対に起きないと断言して大外れ） | $1.00$ | $1.00$ | $\infty$（無限大！） |

- **欠点1（大外れに甘すぎる）**：$1 - p$ や $(1 - p)^2$ では「確率1%」と「確率0.01%」の罰が `0.99` vs `0.9999` でほとんど差がつかず、「絶対に起きない（$p=0$）」と断言して外してもたかだか `1.0` の減点で済んでしまいます。一方 $-\log p$ は $p$ が0に近づくほど急激に重い罰を与え、自信満々な大外れを徹底的に矯正します。
- **欠点2（同時確率の積と一致しない）**：ここで**総乗記号 $\prod$（パイ）**を押さえておきましょう。第3章の $\sum$（全部足す）の掛け算版で、$\prod_{i=1}^{N} p_i = p_1 \times p_2 \times \dots \times p_N$（Pythonでは `prod = 1.0` から `for p in probs: prod *= p`）を意味します。$1 - p$ の平均を小さくしても、文章全体の同時確率 $\prod_{i=1}^N p_i$ を最大にすることとは一致しません。**$-\log p$ の平均だけを小さくすることが、同時確率 $\prod_{i=1}^N p_i$ の最大化（最尤推定）と数学的にぴったり一致**します（$-\frac{1}{N}\log \prod_{i=1}^N p_i = -\frac{1}{N}\sum_{i=1}^N \log p_i$）。
- **先取りの利点（第8章）**：第3章の `softmax` と $-\log p$ を組み合わせると、学習に使う微分（勾配）が $p - y$ という最もシンプルで壊れにくい形になります！

**③ 記号の読み方表**：
| 記号 | 読み方・役割 |
|---|---|
| $L$ | 損失（Loss）。小さいほど良い（0が完璧） |
| $-$ | 負の対数 $\log p_i \le 0$ の符号を反転し、0以上の「驚き」にする |
| $\frac{1}{N}$ | ペア総数 $N$ で割り、文章の長さに依らない「1文字あたりの平均」にする |
| $\sum_{i=1}^{N}$ | $i=1$ から $N$ まで全ペアの対数確率を足し合わせる（第3章の $\sum$） |
| $p_i$ | $i$ 番目の正解文字にモデルが割り当てた予測確率 $P(\text{次}_i \mid \text{今}_i)$ |

**④ 小さな数字で手計算**：3ステップの正解確率が $p = [0.5,\; 0.5,\; 1.0]$ のとき（$\log 0.5 \approx -0.693,\; \log 1.0 = 0$）：
$$L = -\frac{1}{3}\bigl(\log 0.5 + \log 0.5 + \log 1.0\bigr) \approx -\frac{1}{3}(-0.693 - 0.693 + 0) = \frac{1.386}{3} = 0.462$$

**⑤ Pythonの最小コードで検算**：
```python no-run
import math
probs = [0.5, 0.5, 1.0]
loss = -sum(math.log(p) for p in probs) / len(probs)
print(round(loss, 3))   # 0.462
```

#### 💡 理論のポイント：損失は「驚き」の平均

$-\log p$ は、確率 $p$ のことが起きたときの **驚きの大きさ** と考えられます。

天気予報に例えてみましょう。
- 「明日は100%晴れ」と予報して本当に晴れたら、驚きはゼロです（$-\log 1 = 0$）
- 「降水確率50%」と言われて雨が降ったら、コイントスが裏だったくらいで少し驚きます（$-\log 0.5 \approx 0.69$）
- 「降水確率1%」と言われて雨が降ったら、「えっ、降ったの！？」とかなり驚きます（$-\log 0.01 \approx 4.6$）
- 「降水確率0%（絶対に雨は降らない）」と断言していたのに土砂降りになったら、「嘘でしょ！？」と無限大にショックを受けます（$-\log 0 = \infty$）

損失は、モデルが検証データを1文字ずつ読みながら感じた「驚き」の平均です。良いモデルほど、次の文字を正しく予想できているので、驚きが少ない。だから **損失は小さいほど良い** のです。ゴルフのスコアと同じですね。

ちなみに、この量は **負の対数尤度**（negative log-likelihood、NLL）や、**交差エントロピー**（cross-entropy）とも呼ばれます。より一般的には、真の確率分布 $P$ とモデルの予測分布 $Q$ のあいだの交差エントロピーは

$$H(P, Q) = -\sum_{x} P(x)\log Q(x)$$

と定義されます。言語モデルで「正解文字 $y$ だけ確率 $P(y)=1$、他の文字はすべて $0$」（one-hot分布）のときは、和の中で正解文字の項だけが残り $H(P, Q) = -\log Q(y)$ となって負の対数尤度とぴったり一致します。また、真の分布自身の不確かさである **エントロピー** $H(P) = -\sum_{x} P(x)\log P(x)$ と、2つの分布のズレを測る **KLダイバージェンス** $D_{\mathrm{KL}}(P \| Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)}$（第22章・[付録D](https://sakunokyumei.github.io/llm-research-textbook/d-math.html)）を使うと、$H(P, Q) = H(P) + D_{\mathrm{KL}}(P \| Q)$ という関係が成り立ちます。$H(P)$ はデータ固有の定数なので、交差エントロピーを最小化することは、モデルの予測分布 $Q$ を真の分布 $P$ に近づけることと同じ意味になります。

驚きの大きさをグラフにしてみましょう。グラフを描く道具 `matplotlib` を使います（日本語を表示するための追加の道具も入れます）。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

ps = [i / 100 for i in range(1, 101)]
plt.figure(figsize=(6, 3.5))
plt.plot(ps, [-math.log(p) for p in ps])
plt.xlabel("モデルが予想した確率 p")
plt.ylabel("驚き  -log p")
plt.title("確率が低いことが起きるほど、驚きは大きい")
plt.grid(alpha=0.3)
plt.show()

![確率pと驚き-log pのグラフ。pが1に近いと0、pが0に近づくと急に大きくなる](https://sakunokyumei.github.io/llm-research-textbook/img/figs/05-loss/cell7_0.png)

それでは、訓練データと検証データで損失を計算してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def bigram_loss(data, prob_fn):
    total = 0.0
    for a, b in zip(data, data[1:]):
        p = prob_fn(a, b)
        if p == 0:
            return float("inf")       # 確率0の出来事 → 驚きは無限大
        total += -math.log(p)
    return total / (len(data) - 1)

print("訓練データの損失：", bigram_loss(train_text, prob))
print("検証データの損失：", bigram_loss(val_text, prob))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
訓練データの損失： 3.010172965334805
検証データの損失： inf
```

</details>

**ミナト**：検証データの損失が `inf`……無限大？

**ボンゴ先生**：検証データには、訓練データで**一度も見たことがない**文字のペアが出てくるからだ。数えるだけのモデルは、見たことのないペアに確率0をつける。そして、確率0のことが起きたら、驚きは無限大。モデルは「絶対にありえない！」と言い張っていたのに、起きてしまったわけだ。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
unseen = sum(1 for a, b in zip(val_text, val_text[1:]) if pairs[(a, b)] == 0)
print("検証データのペア：", len(val_text) - 1, "個のうち、訓練で見ていないもの：", unseen, "個")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
検証データのペア： 16246 個のうち、訓練で見ていないもの： 1315 個
```

</details>

<details><summary><b>基礎の補足：理解のための最小コード：`float("inf")` とゼロ確率ガード</b>（クリックで開く）</summary>

Pythonで `math.log(0)` を直接呼ぶと `ValueError: math domain error` で強制終了してしまいます。そこで `if p == 0: return float("inf")` で先にガードしています。

| 式 | 値 | 型 | 性質 |
|---|---|---|---|
| `x = float("inf")` | `inf`（正の無限大） | `float`（文字列ではない！） | どんな有限の `float` より大きい（`x > 1e300` は `True`） |
| `math.isinf(x)` | `True` | `bool` | 無限大かどうか安全に判定する標準関数 |
| `math.log(0)` | `ValueError` | 例外でクラッシュ | そのため `p == 0` は必ず事前に分岐する |

```python no-run
import math
inf = float("inf")
print(type(inf), inf > 1e300, math.isinf(inf))   # <class 'float'> True True
```

</details>

#### 📝 確認問題（`ch05-q06` · Lv.3 · なぜ 1-p や (1-p)^2 ではなく -log p を損失に使うのか）

正解トークンにモデルが割り当てた確率を $p$ とするとき、素朴な「外れ度合い」である $1 - p$ や $(1 - p)^2$ ではなく **$-\log p$**（負の対数尤度・交差エントロピー）を言語モデルの損失に採用する理由として**最も適切**なものはどれですか？

- **a.** Pythonの仕様上、浮動小数点数の引き算や2乗より `math.log` しか正確な損失計算に使えないから
- **b.** 素朴な $1 - p$ は正解確率が $p = 1$（完璧な予測）のときに $0$ にならず、基準点がずれてしまうから
- **c.** $-\log p$ はどんな確率 $p$ を入れても必ず $0$ から $1$ の間に収まり、外れ値で発散する心配がないから
- **d.** ① $p \to 0$ の大外れに重い罰を与えられ、②平均の最小化が同時確率の積の最大化と一致するから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：正解に `p = 0.01` をつけた場合と `p = 0.0001` をつけた場合で、$1 - p$ と $-\log p$ の罰の増え方を比べてみましょう。
- **ヒント 2**：$1 - p$ は最大1止まりですが、$-\log p$ は $p \to 0$ で $\infty$ へ急増し、さらに $\sum (-\log p_i) = -\log(\prod p_i)$ で同時確率最大化に直結します。
- **正解**：`d`

**考え方**：$1 - p$ では「1%と思ったのに当たった（`0.99`）」と「0.01%と決めつけて外した（`0.9999`）」の罰がほぼ同じになってしまいます。$-\log p$ なら `4.6` vs `9.2` と2倍の罰になり、さらに $\sum (-\log p_i) = -\log(\prod p_i)$ として同時確率最大化（最尤推定）に直結します。

1. 大外れへの罰：$p=0.01 \to 4.61$、$p=0.0001 \to 9.21$、$p=0 \to \infty$。
1. 同時確率との一致：$-\frac{1}{N}\sum_{i=1}^N \log p_i = -\frac{1}{N}\log\left(\prod_{i=1}^N p_i\right)$。
1. さらに第8章で見るように、softmax と組み合わせたときの微分（勾配）が $p - y$ という最もシンプルで学習しやすい形になる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q06

</details>

#### 📝 確認問題（`ch05-q07` · Lv.2 · 損失 L = -(1/N) Σ log p_i の手計算）

4ステップの予測で、正解文字に対するモデルの予測確率がそれぞれ $p_1 = 0.5, \; p_2 = 0.25, \; p_3 = 0.5, \; p_4 = 1.0$ でした。$\log 0.5 \approx -0.693, \; \log 0.25 \approx -1.386, \; \log 1.0 = 0$ として、この4ステップの平均損失 $L = -\dfrac{1}{4}\sum_{i=1}^{4} \log p_i$ を小数第3位まで求めてください（誤差 $\pm 0.01$ まで正解）。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず各ステップの「驚き」$-\log p_i$（正の数）を4つ求め、その平均をとります。
- **ヒント 2**：4つの驚き `0.693`, `1.386`, `0.693`, `0` をすべて足し合わせ、ステップ数 `4` で割りましょう。
- **正解**：`0.693`

**考え方**：各ステップの驚き $-\log p_i$ は $0.693, 1.386, 0.693, 0$ なので、合計は $2.772$、4で割った平均は $2.772 / 4 = 0.693$ です。

1. ステップ1・3：$-\log(0.5) \approx 0.693$。
1. ステップ2：$-\log(0.25) \approx 1.386$（$0.5^2$ なのでちょうど2倍の驚き）。
1. ステップ4：$-\log(1.0) = 0$。
1. 平均：$L = (0.693 + 1.386 + 0.693 + 0) / 4 = 2.772 / 4 = 0.693$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q07

</details>

*(この節の追加問題 `ch05-q08`, `ch05-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/05-loss.html で自動採点・復習できます)*

## 「絶対」を言わないモデル：スムージング

対策はシンプルです。**どのペアにも、ほんの少しだけ回数を足しておく**のです。見たことがないペアにも「0.01回くらいは見たかもね」とおまけをつける。これを **スムージング**（smoothing、なめらかにする）と言います。

$$P(b \mid a) = \frac{\text{「ab」の回数} + k}{\text{「a」の回数} + k \times V}$$

$V$ は語彙の数です。分母にも $k \times V$ を足すのは、確率の合計をちゃんと1にするためです（全部の $b$ に $k$ ずつ足したので）。

#### 📐 数学のポイント：小さな手計算で確かめる（$V = 4$、$k = 1$）

語彙が `あ, い, う, え`（$V = 4$）だけの世界で、「あ」の出現が6回（内訳：`あい:3, あう:2, ああ:1, あえ:0`）だったとします。全4文字に $k = 1$ ずつ足すと、分母も $k \times V = 1 \times 4 = 4$ 増えて $6 + 4 = 10$ になります。

| 次の文字 $b$ | 元の回数 | $+k$ 後の分子 | スムージング後の確率 $P(b \mid \text{あ})$ |
|---|---|---|---|
| い | 3 | $3+1=4$ | $4/10 = 0.4$ |
| う | 2 | $2+1=3$ | $3/10 = 0.3$ |
| あ | 1 | $1+1=2$ | $2/10 = 0.2$ |
| え（未出現！） | 0 | $0+1=1$ | $1/10 = 0.1$（0にならない！） |
| **合計** | **6** | **$6 + 4 = 10$** | **$0.4 + 0.3 + 0.2 + 0.1 = 1.0$** |

もし分母に $+k \times V$ を忘れると合計が $10/6 = 1.67$ になって確率分布が壊れてしまいます。分母の $k \times V$ は「全 $V$ 文字に $k$ ずつ足した増分を帳尻合わせする項」なのです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
V = len(set(text))

def make_smoothed(k):
    def p(a, b):
        return (pairs[(a, b)] + k) / (firsts[a] + k * V)
    return p

for k in [1, 0.1, 0.01, 0.001]:
    p_k = make_smoothed(k)
    print(f"k={k:<6} 訓練 {bigram_loss(train_text, p_k):.3f}   検証 {bigram_loss(val_text, p_k):.3f}")

> **✨ コツ：補足：語彙サイズ `V` を全テキストから作っている理由**
>
> この教科書の前半（第5〜13章）では、文字トークナイザーの簡潔さを優先して全テキスト `text` から語彙 `chars`・`V` を作っています（検証データにしか出ない文字があっても `KeyError` で止まらないようにするためです）。実務や本格的な研究では検証データの情報が一切漏れないよう、語彙やBPEトークナイザー（第14章）も**訓練データだけ**から構築し、未知語はバイト単位や `<unk>` で扱うのが原則です。

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
k=1      訓練 4.303   検証 4.385
k=0.1    訓練 3.488   検証 3.748
k=0.01   訓練 3.139   検証 3.581
k=0.001  訓練 3.033   検証 3.652
```

</details>

> **🐍 Python・PyTorchのポイント：関数を作る関数**
>
> `make_smoothed(k)` は、`k` の値を覚えた確率関数 `p` を作って返す関数です。関数も変数のように受け渡しできるのがPythonの面白いところです。今は「`k` ごとに別の確率関数を作っている」と分かればOKです。

<details><summary><b>基礎の補足：理解のための最小コード：クロージャ（関数を返す関数 `make_smoothed(k)`）</b>（クリックで開く）</summary>

「関数の中で別の関数 `def p(...)` を作り、その関数自体を `return p` で返す」仕組みを **クロージャ（closure）** と呼びます。外側の引数 `k` を内側の関数が記憶し続けるのがポイントです。

| ステップ | コード | 返る値・型 | 起きていること |
|---|---|---|---|
| ① 工場を呼ぶ | `p_k = make_smoothed(0.01)` | `function`（まだ確率計算はしない！） | `k=0.01` を覚えた新しい関数 `p` が作られる |
| ② 渡す | `bigram_loss(val_text, p_k)` | 関数をそのまま引数 `prob_fn` に渡す | `bigram_loss` の中で毎回 `p_k(a, b)` が呼ばれる |
| ③ 実行 | `p_k("先", "生")` | `float`（約 `0.855`） | 記憶していた `k=0.01` を使って確率を計算する |

最小の例で動きを確かめましょう：
```python no-run
def make_adder(k):
    def f(x):
        return x + k     # 外側の k を覚えている！
    return f             # 関数そのものを返す（f() と呼ばない点に注意）

add2 = make_adder(2)     # type(add2) は <class 'function'>
add10 = make_adder(10)
print(add2(5), add10(5)) # 7 15
```

</details>

#### 💡 理論のポイント：ハイパーパラメータは検証データで選ぶ

`k` は小さいほど**訓練**の損失は良くなります（数えた結果に忠実になるので）。でも**検証**の損失は、`k=0.01` がいちばん良く、小さすぎても大きすぎても悪化しています。

- `k` が大きすぎる：おまけが多すぎて、せっかく数えた知識が薄まる（「先」の次の「生」まで、他の文字と大差なくなる）
- `k` が小さすぎる：見たことのないペアへの驚きが大きすぎる

| 種類 | 誰が決める？ | 例（この章〜今後の章） |
|---|---|---|
| **パラメータ** | **訓練データ**から自動で決まる | bigramの回数表・確率表、ニューラルネットの重み行列（第6章） |
| **ハイパーパラメータ** | **人間**が事前に決め、**検証データ**で選ぶ | スムージングの `k`、学習率（第7章）、層の数、温度 `T` |

ハイパーパラメータは、**検証データの成績で選ぶ**のが鉄則です。訓練データで選ぶと、「丸暗記」に有利な値（`k=0`）を選んでしまいます。

#### 📝 確認問題（`ch05-q10` · Lv.3 · add-k スムージングの式と手計算（なぜ分母に k×V を足すのか））

語彙数 $V = 5$（`か, き, く, け, こ`）の小さな世界で、訓練データに「か」が15回登場し、次の文字の内訳は `かき: 8回, かく: 5回, かか: 2回, かけ: 0回, かこ: 0回` でした。$k = 1$ のスムージング（ラプラス・スムージング）をしたとき、未出現ペア $P(\text{け} \mid \text{か}) = \dfrac{\text{count}(\text{か}, \text{け}) + k}{\text{count}(\text{か}) + k \times V}$ はいくつになりますか？（小数で入力）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：add-$k$ スムージングでは分子に $k$ を足し、全 $V$ 文字分の合計を1に保つため分母に $k \times V$ を足します。
- **ヒント 2**：分子は `0 + 1`、分母は `15 + 1 × 5` です。分子を分母で割って小数に直しましょう。
- **正解**：`0.05`

**考え方**：全5文字それぞれに1回ずつおまけを足したので分母も $5 \times 1 = 5$ 増えて `20` になります。未出現の「け」も確率 $1 / 20 = 0.05$ をもらい、0にならなくなります。

1. 分子：$\text{count}(\text{か}, \text{け}) + k = 0 + 1 = 1$。
1. 分母：$\text{count}(\text{か}) + k \times V = 15 + 1 \times 5 = 20$。
1. $P(\text{け} \mid \text{か}) = 1 / 20 = 0.05$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q10

</details>

#### 📝 確認問題（`ch05-q11` · Lv.2 · 関数を返す関数（クロージャ make_smoothed(k)）の動き）

本文の `make_smoothed(k)` のように「設定値を覚えた関数を返す関数（クロージャ）」の動きを確認します。次のコードを実行すると何が表示されますか？

```python
def make_adder(k):
    def f(x):
        return x + k
    return f

add2 = make_adder(2)
add10 = make_adder(10)
print(add2(5), add10(5))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`make_adder(k)` は、受け取った `k` の値を記憶した新しい関数 `f` を返します（クロージャ）。
- **ヒント 2**：`add2` は引数に `2` を足す関数、`add10` は引数に `10` を足す関数なので、それぞれに `5` を渡した結果を並べます。
- **期待される出力**：`7 15`

**考え方**：`make_adder(2)` を呼ぶと `k=2` を閉じ込めた関数 `f` が新しく作られて `add2` に代入されます。`add2` と `add10` はそれぞれ独立に自分の `k` を記憶しています。

1. `add2 = make_adder(2)` → 引数に `2` を足す関数。
1. `add10 = make_adder(10)` → 引数に `10` を足す関数。
1. `add2(5) = 7`、`add10(5) = 15` → 出力 `7 15`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q11

</details>

*(この節の追加問題 `ch05-q12`, `ch05-q13` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/05-loss.html で自動採点・復習できます)*

## パープレキシティ：何択で迷っているか

損失「3.581」と言われても、ピンとこないかもしれません。そこで、もう少し直感的な数に変換します。

$$\text{パープレキシティ} = e^{\text{損失}}$$

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print("bigram（k=0.01）のパープレキシティ：", math.exp(3.581))
print("でたらめに選ぶモデルの損失：", math.log(V), " パープレキシティ：", math.exp(math.log(V)))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
bigram（k=0.01）のパープレキシティ： 35.909432330660664
でたらめに選ぶモデルの損失： 7.630946580890459  パープレキシティ： 2061.0
```

</details>

#### 📐 数学のポイント：パープレキシティ $\mathrm{PPL} = e^L$ の導出・手計算・実装・落とし穴

**① なぜ $e^L$ が「平均何択」になるのか？（導出）**
各位置 $i$ で正解に割り当てた確率が $p_i$ なら、その位置の「逆数 $1/p_i$」は「その確率 $p_i$ になる均等な選択肢の数」（例：$p_i = 0.25$ なら $1/0.25 = 4$ 択）を表します。全 $N$ 文字にわたる「選択肢の数の幾何平均」を計算して対数・指数で書き換えると、ちょうど $e^L$ に一致します！

$$\mathrm{PPL} = \left(\prod_{i=1}^{N} \frac{1}{p_i}\right)^{1/N} = \exp\!\left(\frac{1}{N}\sum_{i=1}^{N} \log\frac{1}{p_i}\right) = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N}\log p_i\right) = e^L$$

**② 小さな手計算（均等な例 ＆ ばらつきがある例）**
- 毎回均等に4択（$p = 1/4 = 0.25$）：$L = -\log(1/4) = \log 4 \approx 1.386 \;\Rightarrow\; \mathrm{PPL} = e^{\log 4} = 4$（ちょうど4択）
- 毎回100%的中（$p = 1.0$）：$L = -\log 1 = 0 \;\Rightarrow\; \mathrm{PPL} = e^0 = 1$（迷いゼロの1択）
- **確率がばらつく例**：さきほど手計算した $p = [0.5,\; 0.5,\; 1.0]$ のとき
  - 幾何平均から：$(2 \times 2 \times 1)^{1/3} = 4^{1/3} \approx 1.587$
  - 損失 $L \approx 0.462$ から：$e^{0.462} \approx 1.587$（完全に一致！）

**③ 再利用できる最小のPython実装（ゼロ確率・オーバーフロー対策つき）**
```python no-run
import math

def perplexity(probs):
    if any(p <= 0 for p in probs):
        return float("inf")             # 確率0が1つでもあれば PPL = ∞
    loss = -sum(math.log(p) for p in probs) / len(probs)
    if loss > 700:                      # float64 は exp(709.78...) 超で OverflowError
        return float("inf")
    return math.exp(loss)

print(round(perplexity([0.5, 0.5, 1.0]), 3))   # 1.587
print(perplexity([0.25, 0.25, 0.25, 0.25]))    # 4.0
print(perplexity([0.5, 0.0]))                  # inf
```

> **💡 理論のポイント：パープレキシティの直感と「3つの重要な注意点」**
>
> 2061択から約36択へ。数えるだけで、迷いを約57分の1にできたわけです。ただし、パープレキシティを研究や評価で使うときは、次の3つの落とし穴に必ず注意してください。
>
> 1. **「実質 $K$ 択」のたとえの限界**：$\mathrm{PPL} = 10$ は「どの文字でも常に10個が均等に並んでいる」という意味ではなく、あくまで**幾何平均としての等価な選択肢数**です。たとえば99文字を $p \approx 1.0$ で当てても、1文字だけ $p = 10^{-100}$ と大外れすれば全体の $\mathrm{PPL}$ は一気に $10$ へ跳ね上がります（$p=0$ が1つ混ざるだけで $\mathrm{PPL} = \infty$）。
> 2. **トークナイザー（語彙・分割単位）が違うモデルどうしは直接比べられない**：1文字単位とサブワード（BPE、第14章）単位では1トークンあたりの情報量も $N$ も異なるため、生の $\mathrm{PPL}$ の大小を比べることはできません（第20章で詳しく検証します）。
> 3. **評価データや前処理が違う数字は比べられない**：同じモデルでも、易しい文か難しい文か、ルビ除去などの前処理が違うかで $\mathrm{PPL}$ は大きく変わります。公平な比較には**まったく同じ評価データ・前処理・トークナイザー**が必要です。

#### 📝 確認問題（`ch05-q14` · Lv.2 · パープレキシティ e^L の計算と「何択で迷っているか」の意味）

ある言語モデルの検証データの損失がちょうど $L = \log 10 \approx 2.303$ でした。このモデルの**パープレキシティ**（$\text{PPL} = e^L$）はいくつですか？（整数で入力）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：パープレキシティは $\text{PPL} = e^L$ で定義されます。指数関数 $e^x$ と自然対数 $\log x$ の関係（逆関数）を思い出しましょう。
- **ヒント 2**：損失 $L = \log(\text{ある数})$ を $e^L$ に代入したとき、$e$ と $\log$ は互いに打ち消し合って元の数がそのまま出てきます。
- **正解**：`10` 択

**考え方**：$e^{\log 10} = 10$ です。これはモデルが次の1文字を予測するたびに「実質10択クイズくらいの迷い度合い」であることを意味します。

1. 定義：$\text{PPL} = e^{\text{損失}}$。
1. 損失 $L = \log 10$ を代入：$\text{PPL} = e^{\log 10} = 10$。
1. 意味：毎回平均して10択くらいの中で迷っている。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q14

</details>

## 記録表をつけよう

これから章を進むごとに、新しいモデルを作り、この**検証データの損失**で成績を比べていきます。いわば、モデルの進化の記録表です。

| モデル | 章 | 検証データの損失 | パープレキシティ |
|---|---|---|---|
| でたらめ（一様分布） | 5 | 7.631 | 2061 |
| bigram（数える、k=0.01） | 5 | 3.581 | 35.9 |
| ？？？ | 8 | ？ | ？ |
| ？？？ | 9 | ？ | ？ |
| ？？？ | 13 | ？ | ？ |

どこまで小さくできるでしょうか。ちなみに、第13章のGPTは……それは見てのお楽しみ。

#### 📝 確認問題（`ch05-q15` · Lv.3 · 記録表の読み取りと、損失・パープレキシティ・スムージングの総合説明）

記録表で「でたらめモデル」は検証損失 `7.631`（パープレキシティ `2061`）、「bigram（`k=0.01`）」は検証損失 `3.581`（パープレキシティ `35.9`）でした。この数字の改善が直感的に何を意味するのか、また `k` を `0.001` に下げたときに検証損失が `3.652` へ悪化した理由とあわせて説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：パープレキシティの「何択で迷っているか」という意味と、`k` を小さくしすぎたときの未出現ペアの損失 $-\log p$ を思い出しましょう。
- **ヒント 2**：① 2061択から約36択へ迷いを大幅に絞れたこと、② `k=0.001` では未出現ペアの確率が極小になり $-\log p$ の罰が跳ね上がること、の2点を書きます。
- **解答の目安**：でたらめモデルでは毎回2061文字すべてから等確率で迷っていた（2061択）のに対し、bigramモデルは直前1文字の統計を数えただけで平均36択程度（約57分の1の迷い）まで候補を絞り込めていることを意味する。また `k=0.001` に下げすぎると、訓練データに一度も出なかった検証データのペアに与える確率が極端に小さくなり、$-\log p$ の驚き（罰）が跳ね上がるため検証損失が悪化する。

**考え方**：「損失の数字」をパープレキシティで直感化し、同時に検証データでのトレードオフ（知識の薄まり vs 未出現への驚き）を言葉で説明できれば、第5章の核心は完全に定着しています。

1. パープレキシティ比較：2061択 → 35.9択（迷いが約1/57に減少）。
1. `k` が大きすぎる（`k=1`）：おまけが多くて数えた知識が薄まり悪化（4.385）。
1. `k` が小さすぎる（`k=0.001`）：未出現ペアの確率が極小になり $-\log p$ が大きく増えて悪化（3.652）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html#ex-ch05-q15

</details>

<details><summary><b>もっと深く：情報理論との関係（ビットで数える）</b>（クリックで開く）</summary>

自然対数の代わりに $\log_2$ を使うと、損失の単位は **ビット** になります。「1文字を予測するのに、平均何ビット分の情報が足りないか」という意味で、**エントロピー**という考え方につながります。$\log_2 x = \log x / \log 2$ なので、3.581 ÷ 0.693 ≈ 5.17ビット。つまり私たちのモデルは、次の1文字を当てるのに「Yes/Noの質問をあと約5回したい」状態です。<br>
この考え方は、1948年にクロード・シャノンが作った **情報理論** から来ています。シャノンは英語の1文字あたりのエントロピーを、人間に次の文字を当てさせる実験で推定しました。まさに人間を言語モデルとして使ったわけです。

</details>

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`ValueError: math domain error`**：`math.log(0)` を計算しようとしています。確率0が混ざっていないか確認しましょう（それがこの章の事件です）。
> - **グラフの日本語が「□□□」になる**：`import matplotlib_fontja` を忘れていませんか？ その前の `!pip install -q matplotlib-fontja` も実行しましょう。

**✅ できたらチェック**

- [ ] 訓練データと検証データを分ける理由を説明できる
- [ ] 確率を掛け続けると0になる（アンダーフロー）のを見た
- [ ] 対数で掛け算を足し算に変えられることを確かめた
- [ ] 損失＝驚きの平均、小さいほど良い、と説明できる
- [ ] スムージングの k を検証データで選んだ
- [ ] 記録表の最初の2行を埋めた

> **📝 まとめ**
>
> - データは **訓練用** と **検証用** に分ける。点数は検証データでつける
> - 確率の掛け算はすぐ0になる（アンダーフロー）。**対数**で足し算にする
> - **損失** ＝ $-\log(\text{確率})$ の平均 ＝ 驚きの平均。小さいほど良い
> - 見たことのないものに確率0をつけると損失が無限大。**スムージング**で防ぐ
> - **パープレキシティ** ＝ $e^{\text{損失}}$ ＝ 何択で迷っているか

**ボンゴ先生**：数えるモデルの限界は見えてきた。じゃあ「学習する」モデルを作ろう……と言いたいところだが、その前に道具をそろえる。2061行×2061列の数の表を、Pythonの辞書でちまちま扱っていたら日が暮れる。次の章では、大量の数を一気に計算する道具「テンソル」を手に入れる。GPUの本領発揮もそこからだ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html
- **次の章**：[第6章 数の表を一気に扱う：テンソル](https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html)